# Power Profiling — YOLO26 Edge Devices

**Objective:** Collect USB power draw from the JT-TC66C BLE power meter for each
(model, resolution) condition and merge with the existing Table I timing + MOT data
to produce a complete per-device power profile.

**Prerequisites:**
- TC66C connected between USB-C power supply and the DUT
- TC66C Bluetooth menu ON (Menu 6), no phone/app connected to it
- `DEVICE_PROFILE` env var set to the **target device** profile (e.g., `jetson_nano.yaml`)

## Operating modes

### On-device (RPi 4/5)
The notebook runs **on the DUT itself** (which has built-in BLE).
Inference and power logging run in the same process — fully automated.

→ Run cells 1–5, then 7–10.

### Host-based (Jetson, Arduino, or any DUT without BLE)
The notebook runs **on the host computer** (which has BLE to the TC66C).
The DUT is a separate machine. You coordinate manually:

```
┌──────────────────────┐      BLE      ┌─────────┐      USB-C      ┌─────────┐
│  Host (this notebook)│ ◄──────────── │  TC66C  │ ◄────────────── │   DUT   │
│  - scans TC66C       │               │(power   │                 │(Jetson) │
│  - logs power to CSV │               │ meter)  │                 │         │
└──────────────────────┘               └─────────┘                 └─────────┘
```

**Step-by-step:**
1. Set `DEVICE_PROFILE=jetson_nano.yaml` and run cells 1–4 (config + idle baseline)
2. SSH into the DUT, start inference for one condition
3. Run **cell 6** on the host to log power for that condition (90 s)
4. Wait for cell 6 to finish, then stop inference on DUT
5. Edit `CONDITION_IDX` in cell 6 and **re-run** for the next condition
6. When all conditions are done, run cells 7–10 (statistics + figures)

→ Run cells 1–4, then cell 6 repeatedly, then 7–10.  Skip cell 5.

In [ ]:
import asyncio
import os
from pathlib import Path

import matplotlib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import nest_asyncio

from benchmark.config import DATA_ROOT, RESULTS_RAW, SEQUENCES, SEQ_SUFFIX, IMGSZ_BASE
from benchmark.device_profile import load_profile
from benchmark.tc66c import collect, scan_for_tc66c, summarise_readings

# Patch event loop so asyncio.run() works inside Jupyter
nest_asyncio.apply()

PROFILE = load_profile(os.environ.get("DEVICE_PROFILE"))

# ── Output paths ────────────────────────────────────────────────────────────
POWER_DIR     = RESULTS_RAW.parent / "power" / PROFILE.result_tag
PROFILING_DIR = RESULTS_RAW.parent / "profiling"
FIGURES_DIR   = RESULTS_RAW.parent / "figures"
POWER_DIR.mkdir(parents=True, exist_ok=True)

# ── Measurement parameters ─────────────────────────────────────────────────
IDLE_DURATION_S      = 30.0   # idle baseline window
CONDITION_DURATION_S = 90.0   # max per-condition window (host fallback)
INTERVAL_S           = 1.0    # TC66C poll interval
TRIM_S               = 5.0    # seconds trimmed from each end for statistics
SKIP_EXISTING        = True

# ── IEEE two-column figure style ──────────────────────────────────────────
matplotlib.rcParams.update({
    "font.size": 10, "axes.titlesize": 10, "axes.labelsize": 10,
    "xtick.labelsize": 8, "ytick.labelsize": 8, "legend.fontsize": 8,
    "lines.linewidth": 1.2, "axes.linewidth": 0.8,
})

print(f"Device      : {PROFILE.device_label}")
print(f"Backend     : {PROFILE.backend}")
print(f"Tag         : {PROFILE.result_tag}")
print(f"Power dir   : {POWER_DIR}")

In [ ]:
# ── BLE scan and mode detection ──────────────────────────────────────────
# Auto-detects TC66C via local BLE.  If running on the host for a remote DUT,
# the TC66C is still found locally — MODE must be set manually below.
#
# Override: set MODE_OVERRIDE = "host" to force host-based mode even when
# TC66C is found (e.g., running on desktop for a remote Jetson).

MODE_OVERRIDE = None   # set to "host" to force host-based mode

TC66C_ADDRESS = asyncio.run(scan_for_tc66c(timeout=10.0))
print(f"TC66C found at: {TC66C_ADDRESS}")

if MODE_OVERRIDE == "host":
    MODE = "host"
    print(f"Mode: HOST-BASED (forced) — log power here, run inference on DUT via SSH")
else:
    MODE = "on_device"
    print(f"Mode: ON-DEVICE — inference + power logging in one process")
    print(f"  (Set MODE_OVERRIDE = 'host' above if this machine is NOT the DUT)")

In [ ]:
# ── Condition iterator helpers ───────────────────────────────────────────
# Reuses the pattern from notebook 01: Hailo HEFs encode resolution in the
# filename, all other backends iterate the full model × resolution grid.

def _hef_imgsz(model_path: str) -> int:
    """Derive baked-in resolution from a HEF filename stem."""
    stem = model_path.rsplit(".", 1)[0]
    parts = stem.rsplit("_", 1)
    return int(parts[-1]) if len(parts) == 2 and parts[-1].isdigit() else 640

def _model_stem(model_path: str) -> str:
    """Base model name without extension or resolution suffix.
    yolo26n_576.hef -> yolo26n   |   yolo26n.pt -> yolo26n
    """
    stem = model_path.rsplit(".", 1)[0]
    parts = stem.rsplit("_", 1)
    return parts[0] if len(parts) == 2 and parts[-1].isdigit() else stem

def _iter_conditions(profile):
    """Yield (model_path, imgsz) for every measurement condition."""
    if profile.backend == "hailo":
        for mp in profile.model_variants:
            yield mp, _hef_imgsz(mp)
    else:
        for mp in profile.model_variants:
            for imgsz in profile.resolutions:
                yield mp, imgsz

def _power_csv(model_path: str, imgsz: int) -> Path:
    """CSV path: results/power/{tag}/{stem}_{imgsz}.csv"""
    return POWER_DIR / f"{_model_stem(model_path)}_{imgsz}.csv"

conditions = list(_iter_conditions(PROFILE))
print(f"{len(conditions)} conditions to measure:\n")
for mp, sz in conditions:
    status = "EXISTS" if _power_csv(mp, sz).exists() else "pending"
    print(f"  {_model_stem(mp):12s}  {sz}px  [{status}]")

In [ ]:
# ── Idle baseline measurement ────────────────────────────────────────────
# 30 s with no inference running.  Saved once, reused on subsequent runs.
# Operator: ensure the DUT is idle before running this cell.

IDLE_CSV = POWER_DIR / "idle.csv"

if IDLE_CSV.exists() and SKIP_EXISTING:
    print(f"Idle baseline loaded from {IDLE_CSV}")
else:
    print("Collecting idle baseline (30 s) ...")
    readings = asyncio.run(
        collect(
            TC66C_ADDRESS,
            duration_s=IDLE_DURATION_S,
            interval_s=INTERVAL_S,
            csv_path=IDLE_CSV,
            on_reading=lambda r: print(f"  {r.power_W:.3f} W", flush=True),
        )
    )
    print(f"Done — {len(readings)} samples saved to {IDLE_CSV}")

idle_stats = summarise_readings(pd.read_csv(IDLE_CSV), trim_s=TRIM_S)
IDLE_MEAN_W = idle_stats["mean_W"]
print(f"Idle: mean={IDLE_MEAN_W:.3f} W  std={idle_stats['std_W']:.3f} W  "
      f"peak={idle_stats['peak_W']:.3f} W  ({idle_stats['n_samples']} samples)")

In [ ]:
# ── On-device measurement (RPi 4/5 with built-in BLE) ────────────────────
# Power logging runs in a separate process (edge/power_log.py) to fully
# isolate BLE/D-Bus from the inference workload.  SIGINT stops the logger
# cleanly via its signal handler → stop_event.set() → CSV flushed.
#
# Inference always runs (even if experiment CSVs already exist) because we
# need the workload active to measure power.  Results go to a disposable
# path under results/power/{tag}/_warmup/ so they don't pollute exp data.
#
# Skip this cell if MODE == "host"; run cell 6 instead.

import signal
import subprocess
import time as _time

assert MODE == "on_device", "This cell requires on-device mode.  Use cell 6 for host fallback."

from benchmark.device_profile import try_load_model, resolve_model_path
from benchmark.runner import run_sequence
if PROFILE.backend == "hailo":
    from benchmark.hailo_runner import run_sequence_hailo

POWER_LOG = Path(__file__).resolve().parent.parent / "edge" / "power_log.py" \
    if "__file__" in dir() \
    else Path.cwd().parent / "edge" / "power_log.py"
if not POWER_LOG.exists():
    POWER_LOG = Path("/home/joaocarlos/Developer/yolo26-track-edge-benchmark/edge/power_log.py")

import sys
PYTHON = sys.executable

# Disposable directory for inference output during power measurement
WARMUP_DIR = POWER_DIR / "_warmup"
WARMUP_DIR.mkdir(parents=True, exist_ok=True)

def _stop_logger(proc):
    """Stop the power logger subprocess: SIGINT → wait → SIGKILL."""
    if proc.poll() is not None:
        return  # already exited
    proc.send_signal(signal.SIGINT)
    try:
        proc.wait(timeout=5)
    except subprocess.TimeoutExpired:
        proc.kill()
        proc.wait(timeout=3)

for model_path, imgsz in conditions:
    csv_path = _power_csv(model_path, imgsz)
    stem = _model_stem(model_path)

    if SKIP_EXISTING and csv_path.exists():
        print(f"  {stem} {imgsz}px: skip (exists)")
        continue

    print(f"\n── {stem} @ {imgsz}px ──────────────────────")

    # Launch power logger as a subprocess (fully isolated BLE stack)
    proc = subprocess.Popen(
        [PYTHON, str(POWER_LOG),
         "-a", TC66C_ADDRESS,
         "-o", str(csv_path),
         "-d", "600",
         "-i", str(INTERVAL_S)],
        stdout=subprocess.DEVNULL,
        stderr=subprocess.DEVNULL,
    )
    _time.sleep(3.0)  # BLE connection + first sample

    # Run inference — always execute (power measurement needs active workload)
    if PROFILE.backend == "hailo":
        resolved = resolve_model_path(model_path)
        for seq_name in SEQUENCES:
            seq_dir = DATA_ROOT / f"{seq_name}-{SEQ_SUFFIX}"
            out_csv = WARMUP_DIR / f"{seq_name}_{stem}_{imgsz}.csv"
            run_sequence_hailo(resolved, seq_dir, imgsz=imgsz, out_csv=out_csv)
            print(f"  {seq_name}: done")
    else:
        model, err = try_load_model(model_path, PROFILE.torch_device)
        if err:
            print(f"  SKIP — could not load: {err}")
            _stop_logger(proc)
            continue
        for seq_name in SEQUENCES:
            seq_dir = DATA_ROOT / f"{seq_name}-{SEQ_SUFFIX}"
            out_csv = WARMUP_DIR / f"{seq_name}_{stem}_{imgsz}.csv"
            run_sequence(model, seq_dir, imgsz=imgsz, out_csv=out_csv)
            print(f"  {seq_name}: done")
        del model

    # Stop the power logger and count samples
    _stop_logger(proc)
    n_samples = sum(1 for _ in open(csv_path)) - 1 if csv_path.exists() else 0
    print(f"  Power: {n_samples} samples → {csv_path.name}")

# Clean up disposable inference output
import shutil
shutil.rmtree(WARMUP_DIR, ignore_errors=True)
print("\nDone — warmup inference output cleaned up.")

In [ ]:
# ── Host-based: single-condition power capture ───────────────────────────
# For DUTs without BLE (Jetson, Arduino).  This cell measures ONE condition.
#
# WORKFLOW:
#   1. SSH into the DUT, start inference for this condition
#   2. Set CONDITION_IDX below (0-indexed, see cell 3 for the list)
#   3. Run this cell — it logs power for CONDITION_DURATION_S seconds
#   4. When done, stop inference on the DUT
#   5. Increment CONDITION_IDX and re-run for the next condition
#
# The cell auto-skips conditions that already have a CSV (SKIP_EXISTING).

assert MODE == "host", "This cell requires host-based mode.  Set MODE_OVERRIDE = 'host' in cell 2."

# ──────────────────────────────────────────────────────────────────────────
CONDITION_IDX = 0   # ← change this and re-run for each condition
# ──────────────────────────────────────────────────────────────────────────

model_path, imgsz = conditions[CONDITION_IDX]
csv_path = _power_csv(model_path, imgsz)
stem = _model_stem(model_path)

print(f"Condition {CONDITION_IDX}/{len(conditions)-1}: {stem} @ {imgsz}px")

if PROFILE.backend == "hailo":
    print(f"DUT command:  python -m benchmark.runner --model {model_path}")
else:
    print(f"DUT command:  python -m benchmark.runner --model {model_path} --imgsz {imgsz}")

if SKIP_EXISTING and csv_path.exists():
    print(f"SKIP — {csv_path.name} already exists")
else:
    print(f"\nLogging power for {CONDITION_DURATION_S:.0f} s ...")
    print(f"Ensure inference is RUNNING on the DUT before this cell starts.\n")
    readings = asyncio.run(
        collect(
            TC66C_ADDRESS,
            duration_s=CONDITION_DURATION_S,
            interval_s=INTERVAL_S,
            csv_path=csv_path,
            on_reading=lambda r: print(f"  {r.power_W:.3f} W", flush=True),
        )
    )
    stats = summarise_readings(pd.read_csv(csv_path), trim_s=TRIM_S)
    print(f"\nDone — {len(readings)} samples → {csv_path.name}")
    print(f"  mean={stats['mean_W']:.3f} W  delta={stats['mean_W']-IDLE_MEAN_W:.3f} W")

In [ ]:
# ── Aggregate power statistics ───────────────────────────────────────────
# Read all condition CSVs, apply trim, compute per-condition summary.

power_rows = []

for model_path, imgsz in conditions:
    csv_path = _power_csv(model_path, imgsz)
    if not csv_path.exists():
        print(f"  WARNING: missing {csv_path.name}")
        continue

    stats = summarise_readings(pd.read_csv(csv_path), trim_s=TRIM_S)
    power_rows.append({
        "model":     model_path,
        "imgsz":     imgsz,
        "mean_W":    round(stats["mean_W"], 3),
        "std_W":     round(stats["std_W"], 3),
        "peak_W":    round(stats["peak_W"], 3),
        "delta_W":   round(stats["mean_W"] - IDLE_MEAN_W, 3),
        "n_samples": stats["n_samples"],
    })

power_df = pd.DataFrame(power_rows)
print(f"Power profile — {PROFILE.device_label}\n")
print(power_df.to_string(index=False))

In [ ]:
# ── Merge with Table I and save ──────────────────────────────────────────
# Load existing profiling table, aggregate timing/MOT across sequences
# (power is per-(model, imgsz), not per-sequence), join, compute fps/W.

table1_path = PROFILING_DIR / f"table1_profiling_{PROFILE.result_tag}.csv"

if not table1_path.exists():
    print(f"WARNING: {table1_path.name} not found — run notebook 01 first.")
    print("Saving power-only table instead.")
    out_path = PROFILING_DIR / f"table1_power_{PROFILE.result_tag}.csv"
    out_path.parent.mkdir(parents=True, exist_ok=True)
    power_df.to_csv(out_path, index=False)
    print(f"Saved: {out_path}")
else:
    table1 = pd.read_csv(table1_path)

    # Aggregate timing/MOT across sequences
    agg = (
        table1.groupby(["model", "imgsz"])
        .agg(fps=("fps", "mean"), mota=("mota", "mean"), idf1=("idf1", "mean"))
        .reset_index()
        .round({"fps": 1, "mota": 3, "idf1": 3})
    )

    merged = agg.merge(power_df, on=["model", "imgsz"], how="left")
    merged["fps_per_W"] = (merged["fps"] / merged["mean_W"]).round(2)

    out_path = PROFILING_DIR / f"table1_power_{PROFILE.result_tag}.csv"
    out_path.parent.mkdir(parents=True, exist_ok=True)
    merged.to_csv(out_path, index=False)

    print(f"Saved: {out_path}\n")
    print(merged[["model", "imgsz", "fps", "mota", "mean_W", "delta_W", "fps_per_W"]]
          .to_string(index=False))

In [ ]:
# ── Figure: power draw vs model complexity ───────────────────────────────

MODEL_ORDER  = ["yolo26n", "yolo26s", "yolo26m", "yolo26l", "yolo26x"]
MODEL_LABELS = {"yolo26n": "N", "yolo26s": "S", "yolo26m": "M",
                "yolo26l": "L", "yolo26x": "X"}
IMGSZ_COLORS = {640: "#1f77b4", 576: "#ff7f0e", 512: "#2ca02c",
                448: "#d62728", 384: "#9467bd", 320: "#8c564b"}

fig, ax = plt.subplots(figsize=(3.5, 2.8))

plot_df = power_df.copy()
plot_df["stem"] = plot_df["model"].apply(_model_stem)

for imgsz, grp in plot_df.groupby("imgsz"):
    grp = grp.set_index("stem").reindex(MODEL_ORDER).dropna(subset=["mean_W"])
    x = range(len(grp))
    ax.errorbar(
        x, grp["mean_W"], yerr=grp["std_W"],
        fmt="o-", color=IMGSZ_COLORS.get(imgsz, "grey"),
        label=f"{imgsz}px", markersize=4, capsize=3,
    )

ax.set_xticks(range(len(MODEL_ORDER)))
ax.set_xticklabels([MODEL_LABELS[m] for m in MODEL_ORDER])
ax.set_xlabel("Model variant")
ax.set_ylabel("Power draw (W)")
ax.set_title(f"USB power — {PROFILE.device_label}")
ax.legend(fontsize=7, ncol=2)
ax.grid(axis="y", linewidth=0.4, alpha=0.5)

FIGURES_DIR.mkdir(parents=True, exist_ok=True)
for fmt in ("pdf", "png"):
    path = FIGURES_DIR / f"fig_power_{PROFILE.result_tag}.{fmt}"
    fig.savefig(path, bbox_inches="tight", dpi=300)
    print(f"Saved: {path}")
plt.show()

In [ ]:
# ── Figure: inference efficiency (fps / W) ───────────────────────────────
# Requires the merged table from cell 8.

if "merged" not in dir():
    print("Skipped — Table I merge was not available (run notebook 01 first).")
else:
    fig, ax = plt.subplots(figsize=(3.5, 2.8))

    eff_df = merged.copy()
    eff_df["stem"] = eff_df["model"].apply(_model_stem)

    for imgsz, grp in eff_df.groupby("imgsz"):
        grp = grp.set_index("stem").reindex(MODEL_ORDER).dropna(subset=["fps_per_W"])
        x = range(len(grp))
        ax.plot(
            x, grp["fps_per_W"], "s--",
            color=IMGSZ_COLORS.get(imgsz, "grey"),
            label=f"{imgsz}px", markersize=4,
        )

    ax.set_xticks(range(len(MODEL_ORDER)))
    ax.set_xticklabels([MODEL_LABELS[m] for m in MODEL_ORDER])
    ax.set_xlabel("Model variant")
    ax.set_ylabel("Efficiency (fps / W)")
    ax.set_title(f"Inference efficiency — {PROFILE.device_label}")
    ax.legend(fontsize=7, ncol=2)
    ax.grid(axis="y", linewidth=0.4, alpha=0.5)

    for fmt in ("pdf", "png"):
        path = FIGURES_DIR / f"fig_efficiency_{PROFILE.result_tag}.{fmt}"
        fig.savefig(path, bbox_inches="tight", dpi=300)
        print(f"Saved: {path}")
    plt.show()

## Observations

- Idle baseline: *(fill after run)*
- Delta-W range: *(fill after run)*
- Most efficient condition (fps/W): *(fill after run)*

## Next steps

- Repeat for each remaining device profile; CSVs accumulate per `result_tag`
- Run `notebooks/03_results_figures.ipynb` — merge `table1_power_*.csv` into final paper tables
- Add `mean_W` and `fps_per_W` columns to the paper's Table I